In [ ]:
from pathlib import Path
import torch

from data.create_data import build_local_dataloaders, build_global_dataloaders
import data.global_path_datasets as global_paths

from src.diffusion_pipeline.load_diffusion_models import (
    build_global_local_bundles,
    build_mixed_lora_dora_training_setup,
)
from src.score_net.load_scorenet import load_score_net_safely
from src.loss.local_loss import LDLALocalAgingLoss
from src.loss.global_loss import GlobalAgingLoss
from src.loss.global_aux_bundle import GlobalLossAuxBundle
from src.training.train_aging_model import train_global_local_face_aging
from src.training.mixed_precision import resolve_device, get_effective_amp_dtype

In [ ]:
# Runtime
device = resolve_device("auto")
amp_enabled = True
amp_dtype = "bf16"
dtype = get_effective_amp_dtype(amp_dtype=amp_dtype, device=device) or torch.float32

run_name = "notebook_global_local_run"
checkpoint_root = "training_checkpoints/notebook_global_local_run"

In [ ]:
# Global data paths
global_paths.GLOBAL_IMAGE_DIR = Path("data/global_extracted")
global_paths.GLOBAL_CSV_PATH = Path("data/ffhq_predictions/ffhq_face_attribute_prompts.csv")

global_paths.DRIVE_ZIPS = [
    Path("/ruta/a/global_1.zip"),
    Path("/ruta/a/global_2.zip"),
    Path("/ruta/a/global_3.zip"),
    Path("/ruta/a/global_4.zip"),
]

In [ ]:
# Load data
local_objects = build_local_dataloaders(
    batch_size=4,
    num_workers=0,
    pin_memory=False,
)

global_objects = build_global_dataloaders(
    batch_size=4,
    num_workers=0,
    pin_memory=False,
)

local_train_loader = local_objects["train_loader"]
global_train_loader = global_objects["train_loader"]

In [ ]:
# Load diffusion models
global_bundle, local_bundle = build_global_local_bundles(
    global_model_id="SG161222/Realistic_Vision_V6.0_B1_noVAE",
    global_vae_id="stabilityai/sd-vae-ft-mse",
    local_model_id="runwayml/stable-diffusion-v1-5",
    local_vae_id=None,
    device=device,
    dtype=dtype,
    print_memory=True,
)

In [ ]:
# Inject adapters + optimizers
mixed_global_bundle, mixed_local_bundle = build_mixed_lora_dora_training_setup(
    global_bundle=global_bundle,
    local_bundle=local_bundle,
    global_adapter_config={
        "adapter_type": "lora",
        "rank": 8,
        "alpha": 8,
        "dropout": 0.0,
        "target_suffixes": ["to_q", "to_k", "to_v", "to_out.0"],
    },
    local_adapter_config={
        "adapter_type": "dora",
        "rank": 16,
        "alpha": 16,
        "dropout": 0.05,
        "target_suffixes": ["to_q", "to_k", "to_v", "to_out.0"],
    },
    optimizer_config={
        "lr": 1e-4,
        "betas": (0.9, 0.999),
        "weight_decay": 1e-2,
    },
    freeze_before_injection=True,
    print_memory=True,
    print_reports=True,
    verbose=True,
)

In [ ]:
# ScoreNet for local score loss
score_net = load_score_net_safely(
    checkpoint_path="models/score net/score_net_best_overall.pt",
    device=str(device),
    dtype=torch.float32,
    base_channels=32,
    dropout=0.15,
    strict=True,
    freeze=True,
)

In [ ]:
# Local loss
local_loss = LDLALocalAgingLoss(
    local_bundle=mixed_local_bundle,
    score_net=score_net,
    lambda_full=1.0,
    lambda_zone=0.25,
    lambda_score=0.05,
    lambda_cycle=0.0,
    device=str(device),
)

In [ ]:
# Global auxiliary losses
global_aux_bundle = GlobalLossAuxBundle(
    device=str(device),
    dtype=torch.float32,
    use_age=True,
    age_model_id="nateraw/vit-age-classifier",
    age_image_size=224,
    use_identity=True,
    identity_pretrained="vggface2",
    identity_image_size=160,
    use_lpips=False,
)

In [ ]:
# Global loss
global_loss = GlobalAgingLoss(
    global_bundle=mixed_global_bundle,
    global_loss_bundle=global_aux_bundle,
    lambda_diff=1.0,
    lambda_id=0.5,
    lambda_age=0.25,
    lambda_delta_age=0.25,
    lambda_perc=0.0,
    device=str(device),
)

In [ ]:
# Training
result = train_global_local_face_aging(
    mixed_local_bundle=mixed_local_bundle,
    mixed_global_bundle=mixed_global_bundle,
    local_train_loader=local_train_loader,
    global_train_loader=global_train_loader,
    local_loss_fn=local_loss,
    global_loss_fn=global_loss,

    device=device,
    amp_enabled=True,
    amp_dtype="bf16",

    run_name=run_name,
    checkpoint_root=checkpoint_root,

    num_epochs=5,
    local_num_epochs=None,
    global_num_epochs=None,
    train_order=("local", "global"),
    train_local=True,
    train_global=True,

    local_grad_accum_steps=4,
    global_grad_accum_steps=4,
    local_grad_clip=1.0,
    global_grad_clip=1.0,

    local_p_full=0.50,
    local_p_score=0.35,
    local_p_zone=0.15,
    local_enable_full=True,
    local_enable_score=True,
    local_enable_zone=True,
    local_p_neutral=0.10,
    local_p_double_full=0.15,

    global_p_diff=0.55,
    global_p_semantic=0.45,
    global_enable_diff=True,
    global_enable_semantic=True,
    global_semantic_components=("age", "delta_age", "id"),
    global_p_neutral=0.10,
    global_p_double_diff=0.05,
    min_target_age=18,
    max_target_age=90,

    build_schedulers_if_missing=True,
    local_warmup_ratio=0.05,
    global_warmup_ratio=0.05,
    local_min_lr=1e-6,
    global_min_lr=1e-6,
    min_warmup_steps=10,
    max_warmup_steps=None,

    save_latest=True,
    save_best=True,
    save_inference_copy=True,
    local_monitor_key="loss/total",
    global_monitor_key="loss/total",

    enable_gradient_checkpointing_flag=True,
    offload_after_each_branch=True,
    print_memory=True,

    # Para smoke test puedes poner 1 o 2.
    local_max_batches=None,
    global_max_batches=None,

    sample_every_epochs=0,
    sampling_loader_global=None,
    sampling_loader_local=None,
    sampling_output_dir=None,

    inner_print_every=10,
    inner_verbose=False,
    print_first_batch=False,
    verbose=True,
)

result